## Este laboratorio ahora se hace en un Codespace · This lab now runs in a Codespace

**En español.** Lucy ya no responde con reglas fijas: ahora es un agente, como los que estudiamos en el curso. Telegram solo lleva los mensajes; Lucy, que corre en un Codespace, entiende español e inglés con tus propias palabras y recuerda cada conversación. Colab puede interrumpir el proceso si se desconecta. Mantén activos el Codespace y el programa de Lucy mientras escribes desde tu teléfono. El registro muestra quién escribió, cuántos caracteres tenía el mensaje y qué decidió Jev, sin repetir el texto. Usa este enlace en lugar de este cuaderno: [Abrir el laboratorio 10 en un Codespace](https://codespaces.new/profrodai/profrodai-resources?devcontainer_path=.devcontainer/itam-lucy-telegram/devcontainer.json). Necesitas tres secretos: el token de tu bot, tu clave de OpenAI y tu clave de Jev. Los pasos están en la página `NEW-VERSION.md` de esta carpeta.

**In English.** Lucy no longer answers with fixed rules: she is now an agent, as taught in this course. Telegram only carries the messages. Lucy runs in a Codespace, understands Spanish and English in your own words, and remembers each conversation. A Colab disconnect can interrupt the process. Keep both the Codespace and Lucy’s program running while you type from your phone. The log shows who wrote, how long the message was and what Jev decided, without repeating what was typed. Use this link instead of this notebook: [Open lab 10 in a Codespace](https://codespaces.new/profrodai/profrodai-resources?devcontainer_path=.devcontainer/itam-lucy-telegram/devcontainer.json). You need three secrets: your bot token, your OpenAI key and your Jev key. The steps are on the `NEW-VERSION.md` page in this folder.

The notebook below is the old version. You can still read it, but do not use it for the lab.

# Build Lucy’s shop bot on your phone · ITAM · 3 November
**Your goal.** A customer can ask how much ice cream is left. Only Lucy can ask how much to restock. Watch it work, then build your own.

**Rod’s first rehearsal, before class.** Open this notebook in Colab. Have your phone and a Google account ready. Start at step 02 and follow every step through 13, using your phone as the shop owner. These are your first-time installation and token instructions too. Keep the runtime connected and skip the final stop cell. Set vanilla to 2 in step 09 and rerun just that cell. Send `/stock vanilla` to your shop. Once it replies with 2, leave the notebook open. Save the public shop link from step 04.

**In class.** Project the prepared shop chat and do step 01 in that notebook. Stop its listener with the final cell. Open a fresh copy for the guided build and register another bot alongside the students. Hide phone numbers, login codes, BotFather’s token message and /whoami replies. Students can watch your screen before signing in themselves.

**If the demo stalls.** Check the runtime is connected and the listener is running, then send a fresh command. If the network is down, run step 09 as a local calculation. Return to the phone exchange once connected.


## 01 · 00–04 MIN · Watch Lucy answer from her shelf

Rod, project your prepared demo bot’s chat. Anyone can watch before signing in. Ask “How many vanilla tubs does Lucy have?” Send `/stock vanilla`. Show the reply with 2 tubs. Change one number by running this cell in the demo notebook, then send the identical message again. Ask someone to predict the second reply before revealing it. Today each pair will build this exchange and give only Lucy permission to request a restock proposal.

**Checkpoint.** The phone now says 7 tubs. Where did that number come from?

In [ ]:
if 'stock' in globals():
    stock['vanilla'] = 7
    print('Send /stock vanilla again in the demo chat.')
else:
    print('Watch Rod’s phone. Your own build begins in step 02.')

## 02 · 04–13 MIN · Open your two workspaces

Phone: open the official Telegram app from the link below. If you are new, choose Start Messaging, select your country, enter your own phone number and follow Telegram’s verification and profile prompts privately. Stop when you see the chat list. If sign-in takes longer, pair with someone already signed in. Laptop: sign into Google, open the class notebook in Colab (File → Upload notebook if downloaded), save your own copy in Drive, and click Connect. A cell is one runnable block. Press ▶ at its left and read the output directly underneath. Work down the page one cell at a time.

**Checkpoint.** You see Telegram’s chat list and the words “Colab is ready” beneath this cell.

In [ ]:
from IPython.display import Markdown, display
display(Markdown('[Get Telegram](https://telegram.org/apps)'))
print('Colab is ready')

## 03 · 13–18 MIN · Register a new bot

Run the cell and open the verified @BotFather chat. BotFather is Telegram’s account for creating bots. Tap Start, send `/newbot`, then reply with a display name such as Lucy’s Shop. When it asks for a username, try a unique one such as lucy_itam_4821_bot. If taken, change the digits. Rod: hide your phone before BotFather issues the token.

**Checkpoint.** BotFather gives you a public t.me link to your new bot and a private token. Keep this chat private.

In [ ]:
display(Markdown('[Open the official BotFather chat](https://t.me/BotFather)'))

## 04 · 18–22 MIN · Open your shop bot’s chat

Copy the username from the end of your bot’s public t.me link. Run the cell and type that username. Open the resulting link, tap Start, then send `hello lucy` in this new chat. BotFather registers the account. Your shop bot receives customer messages. Switch between these two chats by checking the name at the top.

**Checkpoint.** Your message appears in the shop chat. Silence is expected until you write the reply code.

In [ ]:
import re
username = input('Public bot username only: ').strip().lstrip('@')
assert re.fullmatch(r'[A-Za-z0-9_]+bot', username, re.I), 'Use the username ending in bot.'
display(Markdown(f'[Open your shop @{username}](https://t.me/{username})'))

## 05 · 22–26 MIN · Give Colab the private token

A token is a password that lets code act as this bot. Pause screen sharing. In the BotFather chat copy the token. Run the cell, paste into the hidden input and press Enter. Resume sharing only after the input closes. Each pair uses its own bot and one running notebook. If you accidentally share a token, use BotFather’s /revoke and load the replacement here.

**Checkpoint.** The cell says “Token loaded”. You will verify it with Telegram next.

In [ ]:
from getpass import getpass
TOKEN = getpass('Paste your bot token privately: ').strip()
assert ':' in TOKEN, 'Copy the full token from BotFather and try again.'
print('Token loaded')

## 06 · 26–31 MIN · Ask Telegram which bot you control

An API is a set of requests one program can make to another. Read the last three lines first. Your Python code asks Telegram a question called getMe. Telegram uses the token to decide which bot you mean and returns its public username. Run the cell and compare the printed @username with the public bot username you entered in step 04.

**Checkpoint.** Verified @your_bot. A 401 means Telegram rejected the token. Repeat step 05 and this cell.

In [ ]:
import json, urllib.request, urllib.error, time, threading
def api(method, **params):
    request = urllib.request.Request(
        'https://api.telegram.org/bot' + TOKEN + '/' + method,
        data=json.dumps(params).encode(), headers={'Content-Type': 'application/json'})
    try:
        with urllib.request.urlopen(request, timeout=18) as response:
            result = json.load(response)
    except urllib.error.HTTPError as exc:
        raise RuntimeError(f'Telegram HTTP {exc.code}. Check the token (401) or another poller/webhook (409).') from None
    except OSError:
        raise RuntimeError('Telegram network request failed. Check connectivity and restart the poller.') from None
    if not result.get('ok'):
        raise RuntimeError('Telegram refused the request.')
    return result['result']


bot = api('getMe')
assert bot['username'].lower() == username.lower(), 'This token and username belong to different bots.'
print('Verified @' + bot['username'])

## 07 · 31–39 MIN · Read one message before answering it

Send `hello lucy` again in your shop chat. Run this cell. getUpdates asks Telegram for waiting messages. Look at the three fields the cell extracts. The text tells you what was asked, the sender identifies who asked, and the chat tells you where the answer belongs. The output hides the numeric IDs on the projector. If no matching message appears, confirm the username, resend the words and run again.

**Checkpoint.** Python prints “Received: hello lucy”. Look at your phone. Has anyone answered yet?

In [ ]:
def messages(batch):
    for update in batch:
        message = update.get('message', {})
        if message.get('chat', {}).get('type') == 'private' and 'text' in message:
            yield message

batch = api('getUpdates', timeout=3)
hello = next((m for m in reversed(list(messages(batch))) if m['text'].lower() == 'hello lucy'), None)
assert hello, 'Send hello lucy to this bot and rerun.'
chat_id = hello['chat']['id']
sender_id = hello['from']['id']
print('Received:', hello['text'])

## 08 · 39–44 MIN · Send the first reply yourself

Predict which chat will receive this message. Run the cell once and look at that chat on the phone. sendMessage takes two things you already understand: the destination and the words to send. Ask your partner what would happen if you edited only the words. Try it.

**Checkpoint.** The exact words from Python appear as a new message from your shop bot.

In [ ]:
receipt = api('sendMessage', chat_id=chat_id, text='Hello from Lucy’s shop!')
print('Telegram accepted message', receipt['message_id'])

## 09 · 44–52 MIN · Teach Python to read the shelf

Lucy keeps stock in a Python dictionary. Predict the answer for chocolate, then run the cell. Edit vanilla from 2 to 5, rerun, and explain why chocolate stays the same. Compare strawberry (a product with zero stock) with mango (absent). These checks run entirely inside Colab.

**Checkpoint.** The same function distinguishes an empty shelf from an unknown product.

In [ ]:
stock = {'vanilla': 2, 'chocolate': 12, 'strawberry': 0}
def stock_reply(item):
    if item in stock:
        return f'{item}: {stock[item]} tubs'
    return 'Unknown flavor'
for item in ('vanilla', 'chocolate', 'strawberry', 'mango'):
    print(stock_reply(item))

## 10 · 52–60 MIN · Connect the shelf to incoming commands

A command is ordinary chat text beginning with a slash. Send `/stock vanilla` to your bot and run this cell. Trace one request with your finger: read the message, choose the product, look it up, send the result back. Change the shelf in step 09, send the command again and rerun this step. If several classmates want to test, use one bot per pair.

**Checkpoint.** The phone displays your current vanilla count. Until the next step, each reply requires a click.

In [ ]:
def answer(sender, text):
    command, _, item = text.lower().strip().partition(' ')
    if command == '/stock':
        return stock_reply(item)
    return 'Try /stock vanilla'

if 'offset' not in globals():
    offset = max(u['update_id'] for u in batch) + 1

def handle_waiting():
    global offset
    for update in api('getUpdates', offset=offset, timeout=2):
        for message in messages([update]):
            reply = answer(message['from']['id'], message['text'])
            api('sendMessage', chat_id=message['chat']['id'], text=reply)
        offset = update['update_id'] + 1

handle_waiting()
print('Checked waiting messages. Look in your shop chat.')

## 11 · 60–66 MIN · Keep answering while the notebook runs

Run once. The listener repeats the previous cell’s read-and-reply operation in the background. Send two different stock commands without pressing ▶ again. Ask your partner to open your public link, tap Start and test a third. A 409 means another listener or webhook is using this token. Stop the other listener or create a fresh classroom bot.

**Checkpoint.** Two different phone replies arrive without another click. Colab must stay connected.

In [ ]:
import threading
if 'listener' in globals() and listener.is_alive():
    print('Already listening')
else:
    stop = threading.Event()
    def listen():
        try:
            while not stop.is_set():
                handle_waiting()
        except RuntimeError as error:
            print('Listener stopped.', error)
    listener = threading.Thread(target=listen, daemon=True)
    listener.start()
    print('Listening. Try /stock chocolate.')

## 12 · 66–75 MIN · Give Lucy a permission customers lack

Lucy wants a proposal for enough vanilla to bring the stock up to 8 tubs. Before running, set vanilla to 2 in step 09 and rerun that cell. Each pair should use a fresh bot whose link stayed private until its owner sent hello lucy in step 07. For extra certainty, get your own numeric ID: run the cell, send /whoami privately, then run step 13. The bot reads Telegram’s sender field to check identity. A customer typing “I am Lucy” cannot choose that field.

**Checkpoint.** The owner has not yet been assigned. Both phones get “Owner only” for /restock vanilla.

In [ ]:
owner_id = None
def answer(sender, text):
    command, _, item = text.lower().strip().partition(' ')
    if command == '/whoami':
        return str(sender)
    if command == '/stock':
        return stock_reply(item)
    if command == '/restock':
        if sender != owner_id:
            return 'Owner only'
        if item != 'vanilla':
            return 'No target for that flavor'
        return f'Propose {max(0, 8 - stock[item])} tubs'
    return 'Try /stock vanilla'
print(answer(123, '/restock vanilla'))

## 13 · 75–83 MIN · Try the same request from two phones

Only the intended owner enters the number from their own /whoami reply into the hidden prompt. Both partners now send /restock vanilla to the same bot from different Telegram accounts. Who gets a proposal? Did anything on the shelf change? Show your partner both replies. If working alone, run the two local checks below and label the second-person result simulated.

**Checkpoint.** With 2 tubs on the shelf, Lucy sees “Propose 6 tubs”. A customer sees “Owner only”. Stock stays at 2.

In [ ]:
owner_id = int(getpass('Your own /whoami number (hidden): '))
print('Local owner check:', answer(owner_id, '/restock vanilla'))
print('Local customer check:', answer(owner_id + 1, '/restock vanilla'))
print('Shelf:', stock_reply('vanilla'))

## 14 · 83–88 MIN · Prove you can change the bot

Add a flavor by editing this cell. Ask a partner to predict its reply, then try it from Telegram. Explain aloud how a phone message reached your dictionary and how the answer found its way back. As a final experiment, stop the listener below and send another command. Compare the two outcomes. Save your notebook and a redacted example of the live exchange.

**Checkpoint.** Your new product answers while the listener runs. After you stop it, a new request waits.

In [ ]:
stock['mango'] = 4
print('Send /stock mango from your phone before running the final cell.')

## 15 · 88–90 MIN · Stop the listener and explain the silence

Run this cell and read its output. If it says ‘Finishing the current request’, wait and rerun this same cell. Once it says ‘Stopped’, send `/stock mango` again. Why did the earlier message get a reply while this one waits? Name the part that stopped.

**Checkpoint.** The output says ‘Stopped’, and the new request stays unanswered.

In [ ]:
stop.set()
listener.join(timeout=20)
print('Stopped' if not listener.is_alive() else 'Finishing the current request')

[Telegram apps](https://telegram.org/apps) · [Create a bot](https://core.telegram.org/bots/tutorial) · [Bot API methods](https://core.telegram.org/bots/api)